# Battery Feature Lab: Catenaro–Onori example

Read one raw file with BDS, run BFL, inspect the compact result, retrieve core evidence, and optionally ask GPT-5.6 to interpret the saved JSON records.

Data: Catenaro, Edoardo; Onori, Simona (2021), *Experimental data of three lithium-ion batteries under galvanostatic discharge tests at different C-rates and operating temperatures*, Version 2, [doi:10.17632/kxsbr4x3j2.2](https://doi.org/10.17632/kxsbr4x3j2.2), CC BY 4.0.

Install the optional API client with `uv sync --extra ai`. The API cell reads `OPENAI_API_KEY` from the environment or requests it with a hidden prompt; the key is never written into this notebook.

In [ ]:
import importlib.metadata
import json
from pathlib import Path

import bds

import bfl

repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").is_file() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
dataset_path = Path("examples/data/Catenaro_Onori_2021/NCA_k1_0_05C_05degC.xlsx")
input_path = repo_root / dataset_path
if not input_path.is_file():
    raise FileNotFoundError(input_path)
{
    "input_file": dataset_path.as_posix(),
    "battery-data-standard": importlib.metadata.version("battery-data-standard"),
    "battery-feature-lab": importlib.metadata.version("battery-feature-lab"),
}

In [ ]:
bds_frame, bds_report = bds.read_with_report(
    input_path,
    cycler="auto",
    strict=False,
    keep_raw=True,
    current_sign="charge-positive",
    repair_policy="warn",
    time_sampling_policy="warn",
    current_sign_check="none",
)
{
    "rows": bds_frame.height,
    "columns": bds_frame.columns,
    "cycler": bds_report.to_dict()["cycler"],
    "unmapped_columns": bds_report.to_dict()["unmapped_columns"],
    "time_sampling": bds_report.to_dict()["metadata"]["time_sampling"],
    "semantic_sources": bds_report.to_dict()["metadata"]["semantic_sources"],
}

In [ ]:
result = bfl.analyze(
    input_path,
    output_dir=repo_root / "tmp" / "notebook_outputs" / input_path.stem,
    input_adapter="bds",
    temperature_column="raw:Surface_Temp(degC)",
)
[path.name for path in result.files]

In [ ]:
analysis = json.loads(result.analysis_results_path.read_text(encoding="utf-8"))
metadata = json.loads(result.analysis_metadata_path.read_text(encoding="utf-8"))
validation = json.loads(result.analysis_validation_path.read_text(encoding="utf-8"))
operation_index = analysis["dimensions"]["operation"][0]
{
    "validation_status": validation["status"],
    "output_files": [path.name for path in result.files],
    "output_sizes_bytes": {path.name: path.stat().st_size for path in result.files},
    "dimensions": {
        name: [item["record_type"] for item in items]
        for name, items in analysis["dimensions"].items()
    },
    "operation_sequence": operation_index["attributes"]["operation_sequence"],
    "operation_metrics": operation_index["metrics"],
    "metadata_channels": metadata["channels"],
    "short_window_recomputation": validation["recomputation"]["short_window_recomputation"],
}

In [ ]:
evidence = json.loads(result.analysis_evidence_path.read_text(encoding="utf-8"))
core_types = {
    "response.capacity_aligned_profile",
    "response.current_step_summary",
    "response.relaxation_signature",
}
core_index = {
    item["record_type"]: item
    for item in analysis["dimensions"]["response"]
    if item["record_type"] in core_types
}
core_evidence = {
    record_type: next(
        item
        for item in evidence["records"]
        if item["record_id"] == index["evidence"]["record_id"]
    )
    for record_type, index in core_index.items()
}
{
    record_type: {
        "compact_result": core_index[record_type],
        "source_intervals": record["source_intervals"],
        "method": record["method"],
    }
    for record_type, record in core_evidence.items()
}

In [ ]:
ai_json_paths = (
    result.input_report_path,
    result.analysis_metadata_path,
    result.analysis_results_path,
    result.analysis_evidence_path,
    result.analysis_validation_path,
)
raw_documents = {
    path.name: json.loads(path.read_text(encoding="utf-8"))
    for path in ai_json_paths
}
sensitive_fragments = sorted(
    {
        str(repo_root),
        repo_root.as_posix(),
        str(input_path),
        input_path.as_posix(),
        str(result.output_dir),
        result.output_dir.as_posix(),
        input_path.stem,
    },
    key=len,
    reverse=True,
)

def redact_local_identity(value):
    if isinstance(value, dict):
        return {key: redact_local_identity(item) for key, item in value.items()}
    if isinstance(value, list):
        return [redact_local_identity(item) for item in value]
    if isinstance(value, str):
        for fragment in sensitive_fragments:
            value = value.replace(fragment, "<redacted-local-identity>")
    return value

ai_documents = redact_local_identity(raw_documents)
{
    "documents": list(ai_documents),
    "payload_characters": len(json.dumps(ai_documents, ensure_ascii=False)),
    "local_path_visible": str(repo_root) in json.dumps(ai_documents),
    "source_name_visible": input_path.stem in json.dumps(ai_documents),
}

In [ ]:
import os
from getpass import getpass

from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

prompt = """You are a battery test engineer interpreting machine-readable BDS and Battery Feature Lab JSON.
Use only the supplied documents. Lead with the observed operation, then electrochemical and thermal responses, then evolution or why it is unavailable.
Ground every quantitative statement in a document field or record_id and respect source_intervals, method, applicability, quality flags, and interpretation_limits.
Distinguish measurements from derived metrics. Explain material not_computable results. Do not infer chemistry, named protocol, SOC, SOH, ageing, efficiency, resistance, or mechanism from filenames or missing metadata.
Answer in concise Chinese for a battery engineer."""

client = OpenAI()
response = client.responses.create(
    model="gpt-5.6",
    reasoning={"effort": "medium"},
    text={"verbosity": "medium"},
    instructions=prompt,
    input=json.dumps(ai_documents, ensure_ascii=False),
    max_output_tokens=2500,
    store=False,
)
print(response.output_text)